# TRIBE Studio — Colab GPU backend (for VS Code / Cursor Colab extension)

This notebook runs **the same FastAPI `backend/`** on a **Colab GPU** and opens an **ngrok** HTTPS URL so your **Mac** can set `REMOTE_TRIBE_URL` and keep the light UI there.

**How the Colab extension fits:** Google’s Colab extension runs **notebook cells** on Colab’s cloud machine (GPU). It does **not** by itself publish an HTTP URL for your web app. This notebook adds **uvicorn + ngrok**, which *does* give you a URL.

**Flow:** Run all cells here → copy `REMOTE_TRIBE_URL` → on your Mac: `export REMOTE_TRIBE_URL=...` → start the Mac backend + Vite (see repo `README.md`).

**Do not** set `REMOTE_TRIBE_URL` in this Colab environment (infinite loop).

In [ ]:
# @title 0) GPU check
import torch
print("cuda:", torch.cuda.is_available(), "device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "n/a")

### 1) Put this repo on the Colab VM

- **Option A (recommended):** push your fork to GitHub and set `REPO_URL` below.
- **Option B:** upload `eureka-hacks.zip` whose root contains `backend/`, save as `/content/eureka-hacks.zip`, set `USE_ZIP=True`.

In [ ]:
# @title 1) Clone or unzip
import shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/eureka-hacks.git"  # <-- edit
BRANCH = "main"
ROOT = Path("/content/eureka-hacks")

if ROOT.exists():
    shutil.rmtree(ROOT)

USE_ZIP = False  # True: use /content/eureka-hacks.zip (must contain backend/)

if USE_ZIP:
    z = Path("/content/eureka-hacks.zip")
    if not z.is_file():
        raise FileNotFoundError("Upload eureka-hacks.zip to /content/ or set USE_ZIP=False and fix REPO_URL")
    subprocess.check_call(["unzip", "-q", str(z), "-d", str(ROOT.parent)])
else:
    if "YOUR_GITHUB_USERNAME" in REPO_URL:
        raise RuntimeError("Edit REPO_URL to your fork (or set USE_ZIP=True).")
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(ROOT)])

BACKEND = ROOT / "backend"
assert (BACKEND / "main.py").is_file(), f"Missing backend/main.py under {BACKEND}"
print("OK:", BACKEND)

In [ ]:
# @title 2) Python deps (may take several minutes)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND / "requirements.txt")])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/tribev2.git"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])

In [ ]:
# @title 3) Hugging Face login (gated Llama for TRIBE)
import os, getpass

os.environ.pop("REMOTE_TRIBE_URL", None)
os.environ.pop("TRIBE_DEMO", None)

try:
    from huggingface_hub import login
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])
    from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or getpass.getpass("HF read token (hidden): ").strip()
if not token:
    raise RuntimeError("Missing HF token")
login(token=token, add_to_git_credential=False)
os.environ["HF_TOKEN"] = token
print("HF login OK")

In [ ]:
# @title 4) Stop old processes (re-run friendly)
import subprocess, time
subprocess.run("pkill -f 'uvicorn main:app' || true", shell=True)
subprocess.run("pkill -f '[n]grok' || true", shell=True)
time.sleep(1)

In [ ]:
# @title 5) Start FastAPI (GPU) + ngrok tunnel
import os, subprocess, sys, time, getpass
from pathlib import Path
from urllib.request import urlopen

from pyngrok import conf, ngrok

os.chdir(BACKEND)
os.environ.pop("REMOTE_TRIBE_URL", None)
os.environ["TRIBE_DEVICE"] = os.environ.get("TRIBE_DEVICE", "cuda")

ng_token = getpass.getpass("ngrok authtoken (dashboard.ngrok.com): ").strip()
conf.get_default().auth_token = ng_token
try:
    ngrok.kill()
except Exception:
    pass

tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url.rstrip("/")

log_path = Path("/content/uvicorn.log")
log_f = log_path.open("wb")
proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(BACKEND),
    stdout=log_f,
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(120):
    try:
        urlopen("http://127.0.0.1:8000/api/health", timeout=2)
        ready = True
        break
    except Exception:
        time.sleep(1)

try:
    log_f.flush()
finally:
    log_f.close()

if not ready:
    print("Server did not become ready; log tail:")
    print(log_path.read_text(errors="replace")[-8000:])

analyze_url = public_url + "/api/analyze"
print("\n--- On your Mac (terminal that runs local uvicorn) ---")
print(f'export REMOTE_TRIBE_URL="{analyze_url}"')
print("unset TRIBE_DEMO")
print("# then: cd backend && source .venv/bin/activate && python -m uvicorn main:app --reload --port 8000")
print("\nTunnel base:", public_url)

### VS Code / Cursor **Colab** extension

1. Install Google’s **Colab** extension from the marketplace.
2. Open `notebooks/colab_tribe_backend.ipynb` from this repo.
3. Kernel picker → **Colab** → pick a **GPU** runtime.
4. Run cells in order; keep the Colab session alive while you use the Mac UI.

When the runtime disconnects, the tunnel URL stops working—re-run the start cell for a new URL.